# DSPy — Otimização com `SIMBA`

Neste notebook será demonstrado o uso do otimizador `SIMBA` do DSPy em um problema de **classificação binária de textos**.

Será utilizada a base **Natural Language Processing with Disaster Tweets**, disponibilizada no Kaggle. O objetivo é classificar cada tweet em uma das seguintes categorias:

* `0`: o tweet **não descreve um desastre real**;
* `1`: o tweet **descreve um desastre real**.

O experimento será dividido em três etapas:

1. utilizar o `SIMBA` para gerar programas candidatos a partir do **conjunto de treino**;
2. avaliar os candidatos em um **conjunto de validação separado** e selecionar aquele com maior **F1-score**;
3. comparar o baseline e o programa selecionado utilizando apenas o **conjunto de teste final**.

`SIMBA` significa **Stochastic Introspective Mini-Batch Ascent**.

Em alto nível, o otimizador:

1. amostra mini-batches do conjunto de treino;
2. executa diferentes trajetórias do programa para os mesmos exemplos;
3. identifica exemplos em que existe maior diferença de desempenho entre as trajetórias;
4. cria novos programas candidatos utilizando uma de duas estratégias:
   * adicionar uma demonstração bem-sucedida;
   * gerar uma regra de melhoria por reflexão sobre uma trajetória melhor e uma pior;
5. avalia os novos candidatos e mantém programas promissores ao longo das iterações;
6. ao final, retorna o melhor programa segundo a métrica fornecida ao `SIMBA` e também disponibiliza os candidatos finais em `candidate_programs`.

Como o `SIMBA.compile()` não recebe um `valset`, a validação por F1 será realizada **externamente**, depois da compilação. Isso é importante neste problema porque a métrica utilizada internamente pelo SIMBA é aplicada **por exemplo**, enquanto o F1 é uma métrica **global**, calculada sobre um conjunto de previsões.

A avaliação final utilizará:

* Accuracy;
* Precision;
* Recall;
* F1-score.

A métrica principal para selecionar o candidato e comparar os programas será o **F1-score**.

In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models
import pandas as pd

from typing import Literal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

/home/leonardo/Documentos/github/dspy_studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Primeiro, carregamos as variáveis de ambiente, como a API key, do arquivo `.env` na raiz do projeto.

Serão utilizados dois papéis de modelo durante a otimização:

* `lm`: modelo utilizado pelo classificador e pelas trajetórias avaliadas pelo `SIMBA`;
* `prompt_lm`: modelo utilizado pelo `SIMBA` quando ele gera **regras de melhoria por reflexão**.

Neste exemplo, ambos utilizarão `openai/gpt-5-mini`.

O `SIMBA` requer a dependência opcional `numpy`. Utilizando `uv`, ela pode ser instalada com:

```bash
uv add "dspy[numpy]"
```

O NumPy já pode estar presente no ambiente por ser dependência de outros pacotes, mas o extra acima explicita a dependência requerida pelo otimizador.


In [3]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo utilizado para executar o classificador
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Modelo utilizado pelo SIMBA para gerar regras reflexivas de melhoria.
# Para modelos GPT-5, mantemos temperature=1.0.
prompt_lm = dspy.LM(
    "openai/gpt-5-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
)

# Configura o modelo do classificador como LM padrão do DSPy
dspy.configure(lm=lm)


## 3. Leitura da base de dados

Será utilizada a base da competição **Natural Language Processing with Disaster Tweets**, do Kaggle.

Referência:

https://www.kaggle.com/competitions/nlp-getting-started

Para este experimento são relevantes principalmente duas colunas:

| Coluna   | Descrição                   |
| -------- | --------------------------- |
| `text`   | Texto do tweet              |
| `target` | Classe correta (`0` ou `1`) |

O problema consiste, portanto, em aprender a relação:

`text → target`


In [4]:
df = pd.read_csv('disaster_tweets.csv')
df = df.head(200)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Análise da distribuição das classes

Antes da divisão dos dados, é importante verificar quantos exemplos existem de cada classe.

Além da quantidade absoluta, será analisada a proporção entre tweets classificados como `0` e `1`.

Essa análise é importante porque o **F1-score** considera conjuntamente precisão e recall e é especialmente útil quando existe algum grau de desbalanceamento entre as classes.


In [5]:
df["target"].value_counts()

target
0    103
1     97
Name: count, dtype: int64

In [6]:
df["target"].value_counts(normalize=True)

target
0    0.515
1    0.485
Name: proportion, dtype: float64

## Separação entre treino, validação e teste

Os dados serão divididos em três conjuntos independentes:

* **70% para treino**;
* **15% para validação**;
* **15% para teste**.

Cada conjunto terá uma função diferente:

```text
70% trainset
    ↓
SIMBA gera e otimiza candidatos

15% valset
    ↓
seleção externa do candidato com maior F1

15% testset
    ↓
avaliação final do baseline e do candidato selecionado
```

Essa separação evita selecionar o programa final utilizando os mesmos exemplos empregados durante a otimização.

O `SIMBA.compile()` recebe apenas um `trainset`, portanto somente os **70% de treino** serão passados ao otimizador. O `valset` será utilizado posteriormente para comparar os programas disponíveis em `candidate_programs` pelo F1-score.

O `testset` permanecerá completamente isolado até a avaliação final.

O parâmetro `stratify` será utilizado para manter aproximadamente a mesma proporção entre as classes `0` e `1` nos três conjuntos.

In [7]:
# Primeiro separamos 70% para treino e 30% para validação + teste
df_train, df_temp = train_test_split(
    df[["text", "target"]],
    test_size=0.30,
    random_state=42,
    stratify=df["target"],
)

# Divide os 30% restantes igualmente: 15% validação e 15% teste
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["target"],
)

print(f"Treino:     {len(df_train)} exemplos")
print(f"Validação:  {len(df_val)} exemplos")
print(f"Teste:      {len(df_test)} exemplos")

Treino:     140 exemplos
Validação:  30 exemplos
Teste:      30 exemplos


## Conversão para `dspy.Example`

O DSPy representa exemplos de treino e teste por meio da classe `dspy.Example`.

Neste problema, cada exemplo possui dois campos:

* `text`: entrada fornecida ao modelo;
* `target`: resposta esperada.

A chamada:

`with_inputs("text")`

informa explicitamente ao DSPy que `text` deve ser utilizado como entrada do programa.

Consequentemente, `target` passa a ser tratado como o **label**, ou seja, a resposta esperada para aquele exemplo.

Conceitualmente, cada registro passa a ter a seguinte estrutura:

`entrada: text → saída esperada: target`

In [8]:
def dataframe_para_dspy(dataframe):
    exemplos = []

    for _, row in dataframe.iterrows():
        exemplo = dspy.Example(
            text=row["text"],
            target=int(row["target"]),
        ).with_inputs("text")

        exemplos.append(exemplo)

    return exemplos

In [9]:
trainset = dataframe_para_dspy(df_train)
valset = dataframe_para_dspy(df_val)
testset = dataframe_para_dspy(df_test)

print(f"Trainset: {len(trainset)}")
print(f"Valset:   {len(valset)}")
print(f"Testset:  {len(testset)}")

Trainset: 140
Valset:   30
Testset:  30


In [10]:
trainset[0]

Example({'text': '13,000 people receive #wildfires evacuation orders in California ', 'target': 1}) (input_keys={'text'})

## Definição da tarefa com uma `Signature`

No DSPy, uma `Signature` descreve declarativamente a tarefa que será executada pelo modelo.

A `ClassificarTweet` possui:

* um `InputField` chamado `text`, contendo o tweet;
* um `OutputField` chamado `target`, contendo a classificação.

O tipo:

`Literal[0, 1]`

restringe a resposta esperada às duas classes válidas do problema.

Dessa forma, a Signature define claramente o contrato:

`texto do tweet → 0 ou 1`


In [11]:
class ClassificarTweet(dspy.Signature):
    """
    Classifique o tweet em uma das duas classes possíveis.
    """

    text: str = dspy.InputField(
        desc="Tweet a ser analisado."
    )

    target: Literal[0, 1] = dspy.OutputField(
        desc="Classe prevista."
    )

In [12]:
classificador_base = dspy.Predict(ClassificarTweet)

In [13]:
def avaliar_classificador(programa, dataset, descricao="Avaliando"):
    """
    Executa um programa DSPy sobre um dataset e calcula
    métricas globais de classificação.
    """

    y_true = []
    y_pred = []

    for exemplo in tqdm(dataset, desc=descricao):

        # Executa o programa utilizando somente os campos
        # marcados como entrada pelo with_inputs(...)
        predicao = programa(**exemplo.inputs())

        # Label verdadeiro
        y_true.append(int(exemplo.target))

        # Label previsto pelo DSPy
        y_pred.append(int(predicao.target))

    # Calcula as métricas sobre todo o conjunto
    resultado = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return resultado

In [14]:
resultado_base_validacao = avaliar_classificador(
    classificador_base,
    valset,
    descricao="Baseline no valset",
)

Baseline no valset: 100%|█████████| 30/30 [00:01<00:00, 15.94it/s]


## Baseline no conjunto de validação

Antes da otimização, o classificador original será avaliado no `valset`.

Essa avaliação fornece uma referência para verificar se algum dos candidatos gerados pelo SIMBA melhora o F1 fora do conjunto utilizado na otimização.

O `testset` **ainda não será utilizado**. Ele ficará reservado para a comparação final, depois que o candidato vencedor já tiver sido escolhido.

In [15]:
print(f"F1 baseline no valset: {resultado_base_validacao['f1']:.4f}")

F1 baseline no valset: 0.8148


In [16]:
print(f"Accuracy:  {resultado_base_validacao['accuracy']:.4f}")
print(f"Precision: {resultado_base_validacao['precision']:.4f}")
print(f"Recall:    {resultado_base_validacao['recall']:.4f}")
print(f"F1:        {resultado_base_validacao['f1']:.4f}")

Accuracy:  0.8333
Precision: 0.8462
Recall:    0.7857
F1:        0.8148


## Métrica utilizada durante a otimização

O `SIMBA` precisa de uma métrica aplicada **individualmente a cada exemplo** para atribuir um score às diferentes trajetórias e aos programas candidatos.

Neste problema será utilizada uma métrica simples de **acerto da classe**:

```python
def metrica_simba(example, prediction):
    esperado = int(example.target)
    previsto = int(prediction.target)

    return float(esperado == previsto)
```

A métrica retorna:

* `1.0` quando a classe prevista coincide com a classe esperada;
* `0.0` quando a classificação está incorreta.

Quando agregada sobre vários exemplos, essa métrica corresponde à proporção de acertos, isto é, a uma medida equivalente à **accuracy**.

O **F1-score**, porém, depende conjuntamente de verdadeiros positivos, falsos positivos e falsos negativos de um conjunto inteiro. Por isso, ele não é calculado corretamente como uma métrica independente para cada exemplo.

Neste notebook adotaremos duas etapas:

1. o SIMBA utiliza o acerto individual para gerar e ordenar internamente seus candidatos;
2. os candidatos finais são reavaliados no `valset`, e o programa com maior **F1 global** é escolhido para a avaliação final.

In [17]:
def metrica_simba(example, prediction):
    """
    Métrica utilizada internamente pelo SIMBA.

    Retorna 1.0 quando a classe prevista é igual à classe esperada
    e 0.0 caso contrário.
    """
    esperado = int(example.target)
    previsto = int(prediction.target)

    return float(esperado == previsto)


## Otimização com `SIMBA`

O `SIMBA` (**Stochastic Introspective Mini-Batch Ascent**) é um otimizador que melhora o programa iterativamente a partir da análise de suas próprias execuções.

Para cada etapa de otimização, ele trabalha aproximadamente da seguinte forma:

```text
mini-batch do trainset
        ↓
múltiplas trajetórias por exemplo
        ↓
cálculo da métrica individual
        ↓
identificação de exemplos com resultados contrastantes
        ↓
┌──────────────────────────┬─────────────────────────────┐
│ adicionar demonstração   │ gerar regra por reflexão   │
│ bem-sucedida             │ melhor vs. pior trajetória │
└──────────────────────────┴─────────────────────────────┘
        ↓
novos programas candidatos
        ↓
avaliação no mini-batch
        ↓
pool de candidatos
```

Ao final das etapas, alguns programas representativos da trajetória de otimização são avaliados sobre todo o `trainset`. O SIMBA retorna o melhor segundo sua métrica interna e também disponibiliza os candidatos finais em `candidate_programs`.

Neste notebook usaremos um orçamento menor que os valores padrão para tornar o experimento mais prático com chamadas reais à API:

* `bsize=16`: 16 exemplos em cada mini-batch;
* `num_candidates=3`: três amostragens/candidatos principais por etapa;
* `max_steps=3`: três etapas de otimização;
* `max_demos=4`: no máximo quatro demonstrações armazenadas por preditor.

Depois da compilação, os candidatos serão avaliados separadamente no `valset` pelo F1-score.

In [18]:
BSIZE = 16
NUM_CANDIDATES = 3
MAX_STEPS = 3
MAX_DEMOS = 4

optimizer = dspy.SIMBA(
    metric=metrica_simba,                 # Métrica aplicada a cada previsão
    bsize=BSIZE,                           # Quantidade de exemplos em cada mini-batch
    num_candidates=NUM_CANDIDATES,         # Quantidade de amostragens/candidatos por etapa
    max_steps=MAX_STEPS,                   # Número de etapas de otimização
    max_demos=MAX_DEMOS,                   # Máximo de demonstrações por preditor
    prompt_model=prompt_lm,                # LM usado para gerar regras reflexivas
    num_threads=4,                         # Paralelismo das execuções
    temperature_for_sampling=0.2,          # Exploração ao escolher programas para trajetórias
    temperature_for_candidates=0.2,        # Exploração ao escolher programas-base de candidatos
)


## Compilação do programa

A compilação do `SIMBA` recebe:

* `student`: o programa que será otimizado;
* `trainset`: os exemplos utilizados durante o processo de otimização;
* `seed`: semente utilizada nas escolhas aleatórias internas.

```python
classificador_simba = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    seed=42,
)
```

Diferentemente de alguns outros otimizadores do DSPy, não existe um parâmetro `valset` em `SIMBA.compile()`.

Por isso, o `valset` **não é passado ao SIMBA**. Ele será usado somente depois da compilação para escolher, entre os candidatos finais, aquele que apresenta o maior F1 fora dos dados de treino.

Também é importante garantir que:

```python
len(trainset) >= BSIZE
```

pois o otimizador exige que o conjunto de treinamento tenha pelo menos a quantidade de exemplos definida em `bsize`.

In [19]:
classificador_simba = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    seed=42,
)

2026/09/08 09:15:08 INFO dspy.teleprompt.simba: Starting batch 1 of 3.
2026/09/08 09:15:08 INFO dspy.teleprompt.simba: Sampling program trajectories on 16 examples x 3 samples.


Processed 48 / 48 examples: 100%|█| 48/48 [01:37<00:00,  2.03s/it]

2026/09/08 09:16:45 INFO dspy.teleprompt.simba: Batch 1: Baseline mini-batch score: 0.875

2026/09/08 09:16:45 INFO dspy.teleprompt.simba: Batch 1: Processing bucket #1, with max score 1.0, max-to-min gap 1.0, and max-to-avg gap 0.6666666666666667.
2026/09/08 09:16:45 INFO dspy.teleprompt.simba: Batch 1: Invoking strategy: append_a_demo_
2026/09/08 09:16:45 INFO dspy.teleprompt.simba_utils: Added 1 demos (one each) across all predictors.
2026/09/08 09:16:45 INFO dspy.teleprompt.simba: 

2026/09/08 09:16:45 INFO dspy.teleprompt.simba: Batch 1: Processing bucket #2, with max score 1.0, max-to-min gap 1.0, and max-to-avg gap 0.6666666666666667.
2026/09/08 09:16:45 INFO dspy.teleprompt.simba: Batch 1: Invoking strategy: append_a_demo_
2026/09/08 09:16:45 INFO dspy.teleprompt.simba_utils: Added 1 demos (one each) across all predictors.
2026/09/08 09:16:45 INFO dspy.teleprompt.simba: 

2026/09/08 09:16:45 INFO dspy.teleprompt.simba: Batch 1: Processing bucket #3, with max score 1.0, max-to-m

2026/09/08 09:17:03 INFO dspy.teleprompt.simba_utils: Advice for self: Preprocess: lowercase the tweet, remove or replace URLs (they are noise) but keep the surrounding text. Tokenize on whitespace and punctuation. Detection rules: if the text contains any explicit violent-action keywords or multiword phrases (examples: 'shot', 'shooting', 'gunshot', 'stabbed', 'killed', 'murder', 'assault', 'set ablaze', 'set on fire', 'set afire', 'arson', 'burned', 'burnt', 'bombed', 'explosion', 'wounded', 'victim of', 'shot dead', 'opened fire') then predict target = 1. Multiword phrases (e.g., 'set ablaze', 'set on fire', 'shot before') must be matched as contiguous tokens. Treat the presence of any single explicit violent-action match as sufficient unless strong negation/speculation is present. Negation/speculation handling: if a violence keyword appears within a short negation window (3 tokens before it) containing words like 'not', 'no', 'never', 'didn’t', 'wasn’t', 'alleged', 'allegedly', 're

Processed 64 / 64 examples: 100%|█| 64/64 [02:21<00:00,  2.21s/it]

...

2026/09/08 09:26:19 INFO dspy.teleprompt.simba: Batch 3: Baseline mini-batch score: 0.7291666666666666

2026/09/08 09:26:19 INFO dspy.teleprompt.simba: Batch 3: Processing bucket #1, with max score 1.0, max-to-min gap 1.0, and max-to-avg gap 0.6666666666666667.
2026/09/08 09:26:19 INFO dspy.teleprompt.simba: Batch 3: Invoking strategy: append_a_demo_, having dropped 1 demos per predictor
2026/09/08 09:26:19 INFO dspy.teleprompt.simba_utils: Added 1 demos (one each) across all predictors.
2026/09/08 09:26:19 INFO dspy.teleprompt.simba: 

2026/09/08 09:26:19 INFO dspy.teleprompt.simba: Batch 3: Processing bucket #2, with max score 1.0, max-to-min gap 1.0, and max-to-avg gap 0.33333333333333337.
2026/09/08 09:26:19 INFO dspy.teleprompt.simba: Batch 3: Invoking strategy: append_a_demo_, having dropped 1 demos per predictor
2026/09/08 09:26:19 INFO dspy.teleprompt.simba_utils: Added 1 demos (one each) across all predictors.
2026/09/08 09:26:19 INFO dspy.teleprompt.simba: 

2026/09/08 09:26:


Processed 64 / 64 examples: 100%|█| 64/64 [02:31<00:00,  2.36s/it]

2026/09/08 09:28:50 INFO dspy.teleprompt.simba: Scores after 3 batches: [0.75, 0.875, 0.6875, 0.8125], Best: 0.875

2026/09/08 09:28:50 INFO dspy.teleprompt.simba: VALIDATION: Evaluating 4 programs on the full trainset.



Processed 560 / 560 examples: 100%|█| 560/560 [11:30<00:00,  1.23s

2026/09/08 09:40:20 INFO dspy.teleprompt.simba: Final trainset scores: [0.8428571428571429, 0.9142857142857143, 0.9214285714285714, 0.8285714285714286], Best: 0.9214285714285714 (at index 2)





## Candidatos produzidos pelo `SIMBA`

O programa retornado pelo SIMBA disponibiliza `candidate_programs`, uma lista contendo os programas candidatos finais e seus respectivos scores calculados sobre o `trainset`.

Cada item possui a forma conceitual:

```python
{
    "score": ...,
    "program": ...,
}
```

Esses scores são úteis para entender o comportamento interno do otimizador, mas **não serão usados como critério final de seleção neste notebook**.

A seleção final será feita na próxima etapa, utilizando o **F1-score no `valset`**.

In [20]:
print(f"Quantidade de candidatos finais: {len(classificador_simba.candidate_programs)}")
print(f"Quantidade de etapas registradas: {len(classificador_simba.trial_logs)}")

print("\n=== SCORES INTERNOS NO TRAINSET ===")
for i, candidato in enumerate(classificador_simba.candidate_programs, start=1):
    print(f"Candidato {i}: {candidato['score']:.4f}")

print("\n=== TRIAL LOGS ===")
print(classificador_simba.trial_logs)

Quantidade de candidatos finais: 4
Quantidade de etapas registradas: 3

=== SCORES INTERNOS NO TRAINSET ===
Candidato 1: 0.9214
Candidato 2: 0.9143
Candidato 3: 0.8429
Candidato 4: 0.8286

=== TRIAL LOGS ===
{0: {'train_score': 0.9142857142857143}, 1: {'train_score': 0.9214285714285714}, 2: {'train_score': 0.8285714285714286}}


In [21]:
historico_simba = pd.DataFrame(
    [
        {
            "candidato": i,
            "score_trainset": candidato["score"],
        }
        for i, candidato in enumerate(
            classificador_simba.candidate_programs,
            start=1,
        )
    ]
).sort_values(
    "score_trainset",
    ascending=False,
)

historico_simba

,candidato,score_trainset
0,1,0.921429
1,2,0.914286
2,3,0.842857
3,4,0.828571


## Seleção externa dos candidatos pelo F1 no `valset`

Agora cada programa de `candidate_programs` será executado sobre o **mesmo conjunto de validação**, que não participou da compilação do SIMBA.

Para cada candidato serão calculados:

* Accuracy;
* Precision;
* Recall;
* F1-score.

O candidato com maior **F1 no `valset`** será escolhido como `classificador_otimizado`.

Essa etapa corrige a incompatibilidade entre:

* a métrica individual necessária durante a busca do SIMBA;
* a métrica global que queremos priorizar na tarefa de classificação.

Como o programa baseline faz parte da trajetória de candidatos do SIMBA, a seleção externa também pode, em princípio, preferir um programa mais próximo do baseline caso as alterações propostas pelo otimizador não generalizem bem para a validação.

In [22]:
resultados_candidatos = []

for i, candidato in enumerate(
    classificador_simba.candidate_programs,
    start=1,
):
    programa = candidato["program"]

    resultado = avaliar_classificador(
        programa,
        valset,
        descricao=f"Candidato {i} no valset",
    )

    resultados_candidatos.append(
        {
            "candidato": i,
            "score_trainset": candidato["score"],
            "accuracy_val": resultado["accuracy"],
            "precision_val": resultado["precision"],
            "recall_val": resultado["recall"],
            "f1_val": resultado["f1"],
            "programa": programa,
        }
    )

Candidato 4 no valset: 100%|██████| 30/30 [03:14<00:00,  6.50s/it]


In [23]:
comparacao_validacao = pd.DataFrame(
    [
        {
            "Candidato": r["candidato"],
            "Score trainset": r["score_trainset"],
            "Accuracy val": r["accuracy_val"],
            "Precision val": r["precision_val"],
            "Recall val": r["recall_val"],
            "F1 val": r["f1_val"],
        }
        for r in resultados_candidatos
    ]
).sort_values(
    ["F1 val", "Accuracy val"],
    ascending=False,
)

comparacao_validacao

,Candidato,Score trainset,Accuracy val,Precision val,Recall val,F1 val
3,4,0.828571,0.966667,1.000000,0.928571,0.962963
1,2,0.914286,0.900000,0.923077,0.857143,0.888889
0,1,0.921429,0.833333,0.846154,0.785714,0.814815
2,3,0.842857,0.833333,0.846154,0.785714,0.814815


In [24]:
# Seleciona prioritariamente pelo F1 no conjunto de validação.
# A accuracy é utilizada apenas como critério de desempate.
melhor_candidato = max(
    resultados_candidatos,
    key=lambda r: (r["f1_val"], r["accuracy_val"]),
)

classificador_otimizado = melhor_candidato["programa"]

print(f"Candidato selecionado: {melhor_candidato['candidato']}")
print(f"Score interno no trainset: {melhor_candidato['score_trainset']:.4f}")
print(f"F1 no valset:             {melhor_candidato['f1_val']:.4f}")

Candidato selecionado: 4
Score interno no trainset: 0.8286
F1 no valset:             0.9630


## Inspeção do candidato selecionado

Depois da seleção pelo `valset`, inspecionaremos o estado do programa vencedor.

O SIMBA pode modificar um preditor de duas maneiras principais:

1. acrescentando uma **regra reflexiva** às instruções;
2. adicionando **demonstrações few-shot**.

Por isso, é importante verificar tanto as instruções quanto as demonstrações. A instrução pode permanecer igual à original e, ainda assim, o programa ter sido alterado pelas demos.

In [25]:
print("=== INSTRUÇÃO ORIGINAL ===")
print(classificador_base.signature.instructions)

print("\n=== INSTRUÇÃO DO CANDIDATO SELECIONADO ===")
print(classificador_otimizado.signature.instructions)

print("\n=== DEMONSTRAÇÕES FEW-SHOT SELECIONADAS ===")
print(f"Quantidade de demos: {len(classificador_otimizado.demos)}")

for i, demo in enumerate(classificador_otimizado.demos, start=1):
    print(f"\nDemo {i}:")
    print(demo)

=== INSTRUÇÃO ORIGINAL ===
Classifique o tweet em uma das duas classes possíveis.

=== INSTRUÇÃO DO CANDIDATO SELECIONADO ===
Classifique o tweet em uma das duas classes possíveis.

=== DEMONSTRAÇÕES FEW-SHOT SELECIONADAS ===
Quantidade de demos: 2

Demo 1:
Example({'augmented': True, 'text': 'What a goooooooaaaaaal!!!!!!', 'target': 0}) (input_keys=None)

Demo 2:
Example({'augmented': True, 'text': 'How the West was burned: Thousands of wildfires ablaze in #California alone http://t.co/iCSjGZ9tE1 #climate #energy http://t.co/9FxmN0l0Bd', 'target': 1}) (input_keys=None)


## Avaliação final no `testset`

Somente agora, depois que o candidato vencedor já foi escolhido utilizando o `valset`, o conjunto de teste será utilizado.

Serão avaliados no **mesmo `testset`**:

* o baseline com a instrução original;
* o candidato selecionado externamente pelo maior F1 no `valset`.

O `testset` não participou:

* da geração dos candidatos;
* da otimização interna do SIMBA;
* da escolha do candidato vencedor.

Isso permite que o resultado final seja uma estimativa mais limpa da capacidade de generalização do programa selecionado.

In [26]:
resultado_base = avaliar_classificador(
    classificador_base,
    testset,
    descricao="Baseline no testset",
)

resultado_otimizado = avaliar_classificador(
    classificador_otimizado,
    testset,
    descricao="SIMBA selecionado no testset",
)

Baseline no testset: 100%|████████| 30/30 [00:00<00:00, 64.92it/s]
SIMBA selecionado no testset: 100%|█| 30/30 [03:48<00:00,  7.61s/i


In [27]:
comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Baseline (instrução original)",
            "SIMBA (selecionado pelo F1 no valset)",
        ],
        "Accuracy": [
            resultado_base["accuracy"],
            resultado_otimizado["accuracy"],
        ],
        "Precision": [
            resultado_base["precision"],
            resultado_otimizado["precision"],
        ],
        "Recall": [
            resultado_base["recall"],
            resultado_otimizado["recall"],
        ],
        "F1": [
            resultado_base["f1"],
            resultado_otimizado["f1"],
        ],
    }
)

comparacao

,Modelo,Accuracy,Precision,Recall,F1
0,Baseline (instrução original),0.900000,0.928571,0.866667,0.896552
1,SIMBA (selecionado pelo F1 no valset),0.933333,0.882353,1.000000,0.937500


In [28]:
print("BASELINE")
print(
    classification_report(
        resultado_base["y_true"],
        resultado_base["y_pred"],
        digits=4,
    )
)

print("SIMBA — CANDIDATO SELECIONADO PELO F1 NO VALSET")
print(
    classification_report(
        resultado_otimizado["y_true"],
        resultado_otimizado["y_pred"],
        digits=4,
    )
)

BASELINE
              precision    recall  f1-score   support

           0     0.8750    0.9333    0.9032        15
           1     0.9286    0.8667    0.8966        15

    accuracy                         0.9000        30
   macro avg     0.9018    0.9000    0.8999        30
weighted avg     0.9018    0.9000    0.8999        30

SIMBA — CANDIDATO SELECIONADO PELO F1 NO VALSET
              precision    recall  f1-score   support

           0     1.0000    0.8667    0.9286        15
           1     0.8824    1.0000    0.9375        15

    accuracy                         0.9333        30
   macro avg     0.9412    0.9333    0.9330        30
weighted avg     0.9412    0.9333    0.9330        30



## Salvando o programa selecionado

Neste ponto, `classificador_otimizado` representa o candidato escolhido **externamente pelo F1 no `valset`**, e não necessariamente o programa que o SIMBA havia considerado melhor pelo score interno no `trainset`.

Como o programa possui uma arquitetura simples baseada em `dspy.Predict`, o **State-only Saving** em JSON é adequado:

```python
classificador_otimizado.save("SIMBA.json")
```

Esse arquivo preserva o estado necessário para reutilizar o programa selecionado, incluindo instruções e demonstrações few-shot, mas não a definição Python completa da arquitetura.

Para carregar posteriormente, recriamos a mesma arquitetura (`dspy.Predict(ClassificarTweet)`) e aplicamos `.load()`.

In [29]:
classificador_otimizado.save("SIMBA.json")


In [30]:
# Recria a mesma arquitetura do programa
classificador_carregado = dspy.Predict(ClassificarTweet)

# Carrega o estado otimizado pelo SIMBA
classificador_carregado.load("SIMBA.json")

classificador_carregado


Predict(StringSignature(text -> target
    instructions='Classifique o tweet em uma das duas classes possíveis.'
    text = Field(annotation=str required=True json_schema_extra={'desc': 'Tweet a ser analisado.', '__dspy_field_type': 'input', 'prefix': 'Text:'})
    target = Field(annotation=Literal[0, 1] required=True json_schema_extra={'desc': 'Classe prevista.', '__dspy_field_type': 'output', 'prefix': 'Target:'})
))

In [31]:
tweet = "My phone battery died right before the meeting, what a disaster!"

predicao = classificador_carregado(
    text=tweet
)

print(predicao)

Prediction(
    target=0
)


In [32]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-08T09:54:07.643389]

System message:

Your input fields are:
1. `text` (str): Tweet a ser analisado.
Your output fields are:
1. `target` (Literal[0, 1]): Classe prevista.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "target": "{target}        # note: the value you produce must exactly match (no extra characters) one of: 0; 1"
}
In adhering to this structure, your objective is: 
        Classifique o tweet em uma das duas classes possíveis.


User message:

[[ ## text ## ]]
What a goooooooaaaaaal!!!!!!


Assistant message:

{
  "target": 0
}


User message:

[[ ## text ## ]]
How the West was burned: Thousands of wildfires ablaze in #California alone http://t.co/iCSjGZ9tE1 #climate #energy http://t.co/9FxmN0l0Bd


Assistant message:

{
  "target": 1
}


User message:

[[ ## text ## ]]
My phone b